In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 11.5 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188757 sha256=f6346eb4557bd4a0579f4c249101e66de01829a2e8fdf95e19c292fdcf0d1512
  Stored in directory: /root/.cache/pip/wheels/37/56/51/f27c4eb4bcee969fb4577a5b7d90f9cfe596ebd0e4def65ba5
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61447 sha256=06a7f39d1aa23621220c9aa2092873a5078e6693f6435f6f6d35faabdaf2bad6
  Stored in directory: /root/.cache/pip/whe

In [2]:
# Timer to check how long this code block took to execute
import time
start_time = time.time()

# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
# Trick pytorchvideo into thinking the old deprecated module still exists
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from google.colab import drive

import pytorchvideo.data
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/Train/cow_train_damaged_undamaged"
CLASSES = ['damaged_videos', 'undamaged_videos']
BATCH_SIZE = 2
NUM_FRAMES = 16  # x3d_m defaults to 16 frames
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. ROBUST DATASET DEFINITION ---
labeled_video_paths = []
for i, cls_name in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls_name)
    if not os.path.isdir(cls_path): continue
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.mp4', '.avi', '.mov')):
            labeled_video_paths.append((os.path.join(cls_path, f), {'label': i}))

# X3D Transform Pipeline
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0), # Normalize to 0-1
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]), # X3D specifics
            Resize((256, 256)) # X3D default crop size
        ])
    )
])

# Safely extract 2-second clips
clip_sampler = pytorchvideo.data.make_clip_sampler("uniform", 2.0)

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=labeled_video_paths,
    clip_sampler=clip_sampler,
    transform=video_transform,
    decode_audio=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

# --- 4. MODEL INITIALIZATION ---
print("Downloading X3D-M model from PyTorch Hub...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=True)

# Replace the classification head for X3D
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model = model.to(DEVICE)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n--- Starting X3D Training ---")
model.train()

for epoch in range(EPOCHS):
    running_loss = 0.0
    batches = 0

    for batch_dict in train_loader:
        # X3D expects shape: (Batch, Channels, Time, Height, Width)
        videos = batch_dict['video'].to(DEVICE)
        labels = batch_dict['label'].to(DEVICE)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batches += 1

    avg_loss = running_loss / batches if batches > 0 else 0
    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {avg_loss:.4f}")

# --- 5.1 SAVING THE TRAINED MODEL ---
save_path = "/content/drive/MyDrive/Trained_Models/cow_x3d_damaged_undamaged_TrainedModel.pth"
torch.save(model.state_dict(), save_path)
print(f"Yay! Model safely saved to: {save_path}")

# --- 6. TARGET FILE / INFERENCE WITH THRESHOLD ---
VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

def run_guaranteed_inference(video_path, threshold=0.95):
    """
    Runs inference and intercepts low-confidence predictions
    to label them as 'MISCELLANEOUS'.
    """
    model.eval()
    try:
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)

        # Apply the exact same transform
        video_data = video_transform(video_data)

        # Add batch dimension and send to GPU
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # Extract the values
        confidence_score = conf.item()
        predicted_class = CLASSES[pred_idx.item()].upper()

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")

        # --- THE MISCELLANEOUS INTERCEPT ---
        if confidence_score < threshold:
            print(f"PREDICTED: MISCELLANEOUS (Confidence below {threshold*100}%)")
            print(f"ORIGINAL GUESS: {predicted_class} at {confidence_score*100:.2f}%")
        else:
            print(f"PREDICTED: {predicted_class}")
            print(f"CONFIDENCE: {confidence_score*100:.2f}%")

        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

# Run the inference
run_guaranteed_inference(VIDEO_PATH, threshold=0.70)

# Timer to check how long this code block took to execute
end_time = time.time()
execution_time = end_time - start_time

minutes = int(execution_time // 60)
seconds = execution_time % 60
print(f"Total execution time: {minutes} minutes and {seconds:.2f} seconds")

/usr/local/lib/python3.13/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Mounted at /content/drive
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/X3D_M.pyth" to /root/.cache/torch/hub/checkpoints/X3D_M.pyth


100%|██████████| 29.4M/29.4M [00:00<00:00, 348MB/s]



--- Starting X3D Training ---
Epoch [1/10] Average Loss: 0.4979
Epoch [2/10] Average Loss: 0.3399
Epoch [3/10] Average Loss: 0.3216
Epoch [4/10] Average Loss: 0.3174
Epoch [5/10] Average Loss: 0.3161
Epoch [6/10] Average Loss: 0.3290
Epoch [7/10] Average Loss: 0.3152
Epoch [8/10] Average Loss: 0.3133
Epoch [9/10] Average Loss: 0.3133
Epoch [10/10] Average Loss: 0.3133
Yay! Model safely saved to: /content/drive/MyDrive/Trained_Models/cow_x3d_damaged_undamaged_TrainedModel.pth
------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: UNDAMAGED_VIDEOS
CONFIDENCE: 71.24%
------------------------------
Total execution time: 17 minutes and 28.98 seconds
